# Week 4 — Analyse a protest sequence with a multimodal model

**Research task:** Ask a vision-capable model to describe change across four ordered protest frames while separating visible evidence from interpretation.

**Python introduced:** file paths, strings, ordered lists, dictionaries, Booleans, nested message content and JSON parsing.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session04/session04_multimodal_evidence.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.

For local work, download the complete repository rather than this notebook alone, start it with `uv run jupyter lab`, and follow any `NEXT STEP` printed by the setup cell. The full instructions are in `docs/ENVIRONMENT_SETUP.md` and in the course book's computing chapter.


In [ ]:
SESSION = "session04"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


## What we are studying

Collins asks how a confrontation moves toward, into or away from physical violence. This notebook asks a model to describe observable conduct across selected frames. It does not ask the model to decide whether anyone entered the “tunnel of violence.”

The source is an 18-second protest video from Uttarakhand, India, released by [Himalayanwarrior under CC BY-SA 4.0](https://commons.wikimedia.org/wiki/File:Confrontation_between_Police_and_a_activist_during_a_protest_in_Uttarakhand.webm).

### Load the course settings and choose a route

Run the notebook’s **Prepare the notebook environment** cell first. It installs missing SDKs in Colab, checks the local environment and defines `IN_COLAB`.



`ROOT` is a `Path` pointing to the course folder. `HOSTED_MODEL`, `LOCAL_MODEL` and `ROUTE` are strings. The key prompt appears only when the selected route is OpenRouter. A student using local Ollama does not need to enter a hosted key.

In [ ]:
import json
import os
import sys
from getpass import getpass
from pathlib import Path

try:
    import ollama
    from openrouter import OpenRouter
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "A course SDK is missing. Open a terminal in the complete GenAI_Soc2026 "
        "folder, run 'uv sync --frozen', then reopen this notebook."
    ) from error

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "config" / "course_models.json").exists():
    raise FileNotFoundError(
        "The complete GenAI_Soc2026 repository could not be found. Open this "
        "notebook from the course repository or use its Colab link."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
ROUTE = "openrouter" if IN_COLAB else "ollama"

if ROUTE == "openrouter" and not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Route:", ROUTE)
print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

### Store and display four ordered frames



Each `Path` stores the location of one image file. `frames` is a list. Its order represents time: 4.5, 5.0, 6.0 and 8.5 seconds. `display(...)` shows the evidence before a model is asked to describe it. Each `.exists()` call returns `True` or `False`; `and` requires all four checks to be `True`.

In [ ]:
from IPython.display import Image, display
from src.genai_soc.media import image_to_data_url

IMAGE_DIR = ROOT / "slides" / "session04" / "images"
frame_1 = IMAGE_DIR / "uttarakhand_frame_1_04.5s.png"
frame_2 = IMAGE_DIR / "uttarakhand_frame_2_05.0s.png"
frame_3 = IMAGE_DIR / "uttarakhand_frame_3_06.0s.png"
frame_4 = IMAGE_DIR / "uttarakhand_frame_4_08.5s.png"

frames = [frame_1, frame_2, frame_3, frame_4]

print("frames is a", type(frames))
print("frame_1 is a", type(frame_1))
files_exist = (
    frame_1.exists()
    and frame_2.exists()
    and frame_3.exists()
    and frame_4.exists()
)
print("All four files exist:", files_exist)

display(Image(filename=str(frame_1), width=220))
display(Image(filename=str(frame_2), width=220))
display(Image(filename=str(frame_3), width=220))
display(Image(filename=str(frame_4), width=220))

### Describe the source as a Python dictionary



The dictionary records where the evidence came from, when each frame occurs and what the model did not receive. This implements part of Nassauer and Legewie’s advice about selection and capture. `False` is a Boolean: it records that the call does not include audio.

In [ ]:
source_record = {
    "case": "Uttarakhand protest confrontation",
    "creator": "Himalayanwarrior",
    "source": "Wikimedia Commons",
    "license": "CC BY-SA 4.0",
    "timestamps_seconds": [4.5, 5.0, 6.0, 8.5],
    "audio_used": False,
    "capture_limit": "Four selected stills from an 18-second video",
}

print(source_record)
print("source_record is a", type(source_record))
print("audio_used is a", type(source_record["audio_used"]))

### State the observational task



The prompt operationalizes a boundary between observable conduct and interpretation. Goodwin reminds us that this boundary organizes what the model will treat as salient.

In [ ]:
prompt = """
Examine these protest images in their supplied order.

Give one observation for each frame. Then describe visible changes in distance,
body orientation, gesture, physical contact and intervention by other people.

Do not infer motive, emotion, speech, political identity or responsibility.
State what these selected frames cannot establish.
"""

print(prompt)
print("prompt is a", type(prompt))

### Define the structured return



The schema asks for strings, lists and Booleans. It constrains the form of the answer. It cannot make an observation true.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "frame_observations": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 4,
            "maxItems": 4,
        },
        "visible_changes": {"type": "array", "items": {"type": "string"}},
        "physical_contact_visible": {"type": "boolean"},
        "third_party_intervention_visible": {"type": "boolean"},
        "not_established": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "frame_observations",
        "visible_changes",
        "physical_contact_visible",
        "third_party_intervention_visible",
        "not_established",
    ],
    "additionalProperties": False,
}

print("schema is a", type(schema))

### Send the images through the selected route



OpenRouter receives data URLs over the internet. Ollama receives paths to files on the local machine. Both calls receive the same ordered evidence, prompt and output schema.

In [ ]:
if ROUTE == "openrouter":
    content = [
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_1)}},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_2)}},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_3)}},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_4)}},
    ]
    messages = [{"role": "user", "content": content}]
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(
            model=HOSTED_MODEL,
            messages=messages,
            temperature=0,
            response_format={"type": "json_schema", "json_schema": {
                "name": "protest_sequence",
                "strict": True,
                "schema": schema,
            }},
        )
    raw_output = response.choices[0].message.content
else:
    messages = [{
        "role": "user",
        "content": prompt,
        "images": [str(frame_1), str(frame_2), str(frame_3), str(frame_4)],
    }]
    try:
        response = ollama.chat(think=False,
            model=LOCAL_MODEL,
            messages=messages,
            format=schema,
            options={"temperature": 0},
        )
    except ConnectionError as error:
        raise ConnectionError(
            "Python could not reach Ollama. Start the Ollama application, run "
            "'ollama list' in a terminal, and then rerun this cell."
        ) from error
    raw_output = response.message.content

print("Raw JSON text:", raw_output)
print("raw_output is a", type(raw_output))

### Parse and inspect the model’s claims



`raw_output` is JSON text stored as a string. `json.loads(...)` converts that text into a Python dictionary. Square brackets retrieve one named value from that dictionary.

In [ ]:
description = json.loads(raw_output)

print("description is a", type(description))
print("Frame observations:", description["frame_observations"])
print("Visible changes:", description["visible_changes"])
print("Physical contact visible:", description["physical_contact_visible"])
print("Third-party intervention visible:", description["third_party_intervention_visible"])
print("Not established:", description["not_established"])

### Record the researcher’s assessment



The model’s return and the researcher’s correction now sit in the same record. This makes disagreement visible rather than silently replacing the model output.

In [ ]:
supported_claim = input("One claim clearly supported by the frames: ").strip()
unsupported_claim = input("One unsupported or overstated claim: ").strip()
missing_observation = input("One visible detail the model missed: ").strip()

if not supported_claim or not unsupported_claim or not missing_observation:
    raise ValueError("Complete all three parts of the researcher check.")

researcher_check = {
    "supported_claim": supported_claim,
    "unsupported_or_overstated_claim": unsupported_claim,
    "missing_observation": missing_observation,
}

research_record = {
    "source": source_record,
    "route": ROUTE,
    "model": HOSTED_MODEL if ROUTE == "openrouter" else LOCAL_MODEL,
    "prompt": prompt,
    "raw_output": raw_output,
    "parsed_output": description,
    "researcher_check": researcher_check,
}

print("Saved record fields:", list(research_record.keys()))
print("Researcher check:", research_record["researcher_check"])

### Make one controlled change

Replace only the final frame. Keep the first three frames, prompt, schema, route, model and temperature fixed:



`changed_frames` is a new list, so the original evidence and result remain available for comparison.

In [ ]:
frame_5 = IMAGE_DIR / "uttarakhand_frame_5_11.5s.png"
changed_frames = [frame_1, frame_2, frame_3, frame_5]
changed_timestamps = [4.5, 5.0, 6.0, 11.5]

display(Image(filename=str(frame_5), width=220))
print("Original endpoint:", source_record["timestamps_seconds"][-1])
print("Changed endpoint:", changed_timestamps[-1])

### Make the second call without overwriting the first

This cell repeats the route-specific call using the later endpoint. The repetition is deliberate: functions arrive later in the course, so every operation remains visible here.

In [ ]:
if ROUTE == "openrouter":
    changed_content = [
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_1)}},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_2)}},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_3)}},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_5)}},
    ]
    changed_messages = [{"role": "user", "content": changed_content}]
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        changed_response = client.chat.send(
            model=HOSTED_MODEL,
            messages=changed_messages,
            temperature=0,
            response_format={"type": "json_schema", "json_schema": {
                "name": "protest_sequence",
                "strict": True,
                "schema": schema,
            }},
        )
    changed_raw_output = changed_response.choices[0].message.content
else:
    changed_messages = [{
        "role": "user",
        "content": prompt,
        "images": [str(frame_1), str(frame_2), str(frame_3), str(frame_5)],
    }]
    try:
        changed_response = ollama.chat(think=False,
            model=LOCAL_MODEL,
            messages=changed_messages,
            format=schema,
            options={"temperature": 0},
        )
    except ConnectionError as error:
        raise ConnectionError(
            "Python could not reach Ollama. Start the Ollama application, run "
            "'ollama list' in a terminal, and then rerun this cell."
        ) from error
    changed_raw_output = changed_response.message.content

print("Changed raw JSON text:", changed_raw_output)
changed_description = json.loads(changed_raw_output)
print("Changed result:", changed_description)

### Check the second return and compare the endpoints

`raw_output` and `description` still contain the first return. `changed_raw_output` and `changed_description` contain the second. The next cell stores a separate check and places both records beside your comparison.

In [ ]:
changed_supported_claim = input(
    "One claim clearly supported in the later-endpoint return: "
).strip()
changed_unsupported_claim = input(
    "One unsupported or overstated claim in that return: "
).strip()
changed_missing_observation = input(
    "One visible detail that return missed: "
).strip()
endpoint_comparison = input(
    "How did replacing 8.5 seconds with 11.5 seconds change the account? "
).strip()

if not all([
    changed_supported_claim,
    changed_unsupported_claim,
    changed_missing_observation,
    endpoint_comparison,
]):
    raise ValueError("Complete all four parts of the endpoint comparison.")

changed_source_record = source_record.copy()
changed_source_record["timestamps_seconds"] = changed_timestamps
changed_researcher_check = {
    "supported_claim": changed_supported_claim,
    "unsupported_or_overstated_claim": changed_unsupported_claim,
    "missing_observation": changed_missing_observation,
}
changed_research_record = {
    "source": changed_source_record,
    "route": ROUTE,
    "model": HOSTED_MODEL if ROUTE == "openrouter" else LOCAL_MODEL,
    "prompt": prompt,
    "raw_output": changed_raw_output,
    "parsed_output": changed_description,
    "researcher_check": changed_researcher_check,
}

comparison_record = {
    "original": research_record,
    "later_endpoint": changed_research_record,
    "student_comparison": endpoint_comparison,
}

print("Original visible changes:", description["visible_changes"])
print("Later-endpoint visible changes:", changed_description["visible_changes"])
print("Your comparison:", endpoint_comparison)

## Take-home recording

Run the original sequence and the explicit later-endpoint call. Explain each input, Python type and output. Connect the comparison to Collins and either Goodwin or Nassauer and Legewie. Do not claim that the images establish motive, fear or entry into the tunnel of violence.

Upload the continuous narrated recording by **5:00 p.m. Eastern on the Tuesday before the next class**.